# SongNet Demo: Real-Time Genre Classification

The SongNet paper's goal is genre classification **while a song plays**, but the authors did not build
it (Section 5.4: "Due to limited timeline, we did not implement it yet"). This notebook demonstrates it
with our trained C-RNN:

1. **Live prediction on a test song:** the genre probabilities the model would show at every moment
   of the 30-second clip.
2. **Your own song:** upload any audio file and watch the prediction form.
3. **Accuracy vs. seconds heard:** how accurate the model is after hearing only the first 1, 3, 5, 10 ...
   seconds of each of the 800 test songs.
4. **Speed:** how long the model takes to classify a clip, compared with the clip's length.

**Before running:** run `CRNN.ipynb` first, so that `MyDrive/SongNet/models/crnn_best.pt` exists.
A GPU runtime is faster but not required.

### How the model predicts in real time

The C-RNN outputs 8 genre scores at each of its 161 time steps (one step is about 0.19 s of audio).
The GRU reads the steps left to right, so the scores at second *t* depend only on audio up to *t*,
plus under 0.2 s of look-ahead from the convolution kernels. The song-level prediction is the mean of
the step scores, so **the mean of the steps heard so far is the prediction available at that moment**.

## 1. Setup

In [ ]:
import hashlib
import random
import subprocess
import sys
import time
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Audio, display
from joblib import Parallel, delayed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True  # the trained model is read from Google Drive

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive/SongNet")
    else:
        PROJECT_ROOT = Path("/content/SongNet")
    FMA_DIR = Path("/content/fma")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    FMA_DIR = PROJECT_ROOT / "data"

AUDIO_DIR = FMA_DIR / "fma_small"
MODEL_PATH = PROJECT_ROOT / "models" / "crnn_best.pt"
SPLITS_PATH = PROJECT_ROOT / "data" / "splits.csv"
SPLITS_URL = ("https://raw.githubusercontent.com/soniaa27/"
              "SongNet-Real-time-Music-Classification/main/splits.csv")
RESULTS_DIR = PROJECT_ROOT / "results" / "demo"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_IF_MISSING = IN_COLAB  # never downloads the dataset outside Colab

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device, "| model:", MODEL_PATH)

## 2. Load the trained C-RNN

The model class is identical to the one in `CRNN.ipynb`; its settings and the normalisation
statistics are read from the checkpoint, so the demo uses exactly the trained model.

In [ ]:
class CRNN(nn.Module):
    def __init__(self, n_mels, n_classes, conv_channels, pool_sizes, rnn_hidden, rnn_layers, dropout):
        super().__init__()
        blocks, in_ch, freq = [], 1, n_mels
        for out_ch, pool in zip(conv_channels, pool_sizes):
            blocks += [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                       nn.BatchNorm2d(out_ch),
                       nn.ReLU(inplace=True),
                       nn.MaxPool2d(tuple(pool)),
                       nn.Dropout(dropout)]
            in_ch, freq = out_ch, freq // pool[0]
        self.conv = nn.Sequential(*blocks)
        self.rnn = nn.GRU(in_ch * freq, rnn_hidden, num_layers=rnn_layers, batch_first=True,
                          dropout=dropout if rnn_layers > 1 else 0.0)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(rnn_hidden, n_classes))

    def forward(self, x, return_timesteps=False):
        z = self.conv(x)                                  # (B, C, F, T)
        z = z.permute(0, 3, 1, 2).flatten(2)              # (B, T, C*F)
        z, _ = self.rnn(z)                                # (B, T, H)
        step_logits = self.classifier(z)                  # (B, T, n_classes)
        logits = step_logits.mean(dim=1)                  # mean over timesteps
        return (logits, step_logits) if return_timesteps else logits


if not MODEL_PATH.exists():
    raise FileNotFoundError(f"{MODEL_PATH} not found: run CRNN.ipynb first.")
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
cfg = checkpoint["config"]
CLASS_NAMES = checkpoint["class_names"]
MEL_MEAN = torch.as_tensor(checkpoint["mel_mean"], device=device)[:, None]
MEL_STD = torch.as_tensor(checkpoint["mel_std"], device=device)[:, None]
SR, HOP = cfg["sr"], cfg["hop_length"]
CLIP_SECONDS = cfg["clip_seconds"]

model = CRNN(cfg["n_mels"], len(CLASS_NAMES), cfg["conv_channels"], cfg["pool_sizes"],
             cfg["rnn_hidden"], cfg["rnn_layers"], cfg["dropout"]).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"Loaded epoch {checkpoint['epoch']} | validation accuracy {checkpoint['val_accuracy']:.4f}")
print("Genres:", CLASS_NAMES)

## 3. Test songs and audio

The demo uses songs from the **test split**, which the model never saw during training or model
selection. The split is read from Drive, or from the GitHub repository if it is not on Drive; its
checksum must match the one stored with the model.

In [ ]:
if not SPLITS_PATH.exists():
    SPLITS_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["wget", "-nv", "-O", str(SPLITS_PATH), SPLITS_URL], check=True)
splits = pd.read_csv(SPLITS_PATH)
sha = hashlib.sha256(SPLITS_PATH.read_bytes()).hexdigest()
assert sha == checkpoint["splits_sha256"], "splits.csv differs from the split the model was trained on"
test = splits[splits["split"] == "test"].reset_index(drop=True)
print(f"{len(test)} test songs | splits.csv sha256 = {sha[:8]}... (matches the model)")

if not AUDIO_DIR.exists():
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(f"Place the FMA-small audio in {AUDIO_DIR}")
    FMA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = FMA_DIR / "fma_small.zip"
    print("Downloading fma_small.zip (7.2 GB) ...")
    subprocess.run(["wget", "-nv", "-O", str(zip_path),
                    "https://os.unil.cloud.switch.ch/fma/fma_small.zip"], check=True)
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(FMA_DIR)], check=True)
    zip_path.unlink()


def audio_path(track_id):
    tid = f"{int(track_id):06d}"
    return AUDIO_DIR / tid[:3] / f"{tid}.mp3"


print("Audio ready:", AUDIO_DIR)

## 4. Prediction over time

`mel_from_audio` applies the same preprocessing as training (mono, 22,050 Hz, 30 s, 128-band
log-mel, training-set normalisation). `predict_over_time` returns, for every time step, the genre
probabilities the model would show at that moment: the softmax of the mean of the step scores so far.

In [ ]:
def mel_from_audio(y):
    y = librosa.util.fix_length(y, size=int(SR * CLIP_SECONDS))
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=cfg["n_fft"], hop_length=HOP,
                                         n_mels=cfg["n_mels"])
    return librosa.power_to_db(mel, ref=1.0, top_db=80.0).astype(np.float32)


def normalise(mels):
    x = torch.as_tensor(np.asarray(mels, dtype=np.float32), device=device)
    if x.dim() == 2:
        x = x[None]
    return ((x - MEL_MEAN) / MEL_STD).unsqueeze(1)          # (B, 1, mels, frames)


@torch.no_grad()
def predict_over_time(mel):
    _, steps = model(normalise(mel), return_timesteps=True)  # (1, T, 8)
    steps = steps[0]
    running = steps.cumsum(0) / torch.arange(1, len(steps) + 1, device=device)[:, None]
    probs = running.softmax(dim=1).cpu().numpy()             # (T, 8)
    seconds = (np.arange(len(probs)) + 1) * CLIP_SECONDS / len(probs)
    return seconds, probs


def show_live_prediction(y, title, true_genre=None):
    mel = mel_from_audio(y)
    seconds, probs = predict_over_time(mel)
    final = CLASS_NAMES[int(probs[-1].argmax())]

    print("What the model would show while the song plays:")
    for t in (1, 3, 5, 10, 15, 20, 25, 30):
        i = min(int(np.searchsorted(seconds, t)), len(seconds) - 1)
        g = int(probs[i].argmax())
        print(f"  after {t:2d} s -> {CLASS_NAMES[g]:<13} ({probs[i, g]:.0%})")
    verdict = "" if true_genre is None else (" (correct)" if final == true_genre else " (wrong)")
    print(f"Final prediction: {final}{verdict}" + ("" if true_genre is None else f" | true genre: {true_genre}"))

    fig, (ax_mel, ax) = plt.subplots(2, 1, figsize=(11, 7.5), sharex=True,
                                     gridspec_kw={"height_ratios": [1, 1.2]})
    librosa.display.specshow(mel, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=ax_mel)
    ax_mel.set(title="Mel-spectrogram: what the model hears", xlabel="")
    for g, name in enumerate(CLASS_NAMES):
        highlight = name in (final, true_genre)
        ax.plot(seconds, probs[:, g], lw=2.5 if highlight else 1, alpha=1 if highlight else 0.45,
                label=name + (" (true)" if name == true_genre else ""))
    ax.set(xlabel="seconds of audio heard", ylabel="probability", ylim=(0, 1), xlim=(0, CLIP_SECONDS),
           title=title)
    ax.legend(ncol=4, fontsize=9, loc="upper left")
    fig.tight_layout()
    return fig

## 5. Demo: a song from the test set

Change `TRACK_ID` to any ID from the test split, or leave it as `None` for a random test song.
Play the audio, then read the figure from left to right. The top panel is the mel-spectrogram the
model receives; the bottom panel is what a live display would show at each moment, on the same
time axis.

In [ ]:
TRACK_ID = None  # e.g. 2 or 10; None picks a random test song

row = test.sample(1, random_state=None).iloc[0] if TRACK_ID is None else \
    test[test["track_id"] == TRACK_ID].iloc[0]
path = audio_path(row.track_id)
y, _ = librosa.load(path, sr=SR, mono=True, duration=CLIP_SECONDS)

print(f"Track {row.track_id} | true genre: {row.genre}")
display(Audio(y, rate=SR))
fig = show_live_prediction(y, f"Track {row.track_id}: genre probabilities as the song plays", row.genre)
fig.savefig(RESULTS_DIR / f"live_track_{row.track_id}.png", dpi=150)
plt.show()

## 6. Demo: your own song

Upload any MP3/WAV file. The model was trained on 30-second excerpts from the middle of FMA tracks,
so a 30-second window from the middle of your file is used. Genres outside the 8 FMA genres will be
mapped to the closest of the 8.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        duration = librosa.get_duration(path=name)
        offset = max(0.0, duration / 2 - CLIP_SECONDS / 2)
        y_user, _ = librosa.load(name, sr=SR, mono=True, offset=offset, duration=CLIP_SECONDS)
        print(f"{name}: using {offset:.0f}s to {offset + len(y_user) / SR:.0f}s")
        display(Audio(y_user, rate=SR))
        show_live_prediction(y_user, f"{name}: genre probabilities as the song plays")
        plt.show()
else:
    print("Upload works in Colab. Locally, set y_user = librosa.load('<file>', sr=SR)[0] and call "
          "show_live_prediction(y_user, '<title>').")

## 7. How much audio does the model need?

For each of the 800 test songs, the model hears only the **first k seconds** (the audio after k is
removed, exactly as in a live stream) and we measure test accuracy. The 30-second value is the
model's full test accuracy reported in `CRNN.ipynb`.

This step computes spectrograms for the 800 test songs, which takes a few minutes.

In [ ]:
SECONDS = [1, 2, 3, 5, 10, 15, 20, 30]


def test_mel(track_id):
    y, _ = librosa.load(audio_path(track_id), sr=SR, mono=True, duration=CLIP_SECONDS)
    return mel_from_audio(y).astype(np.float16)


start = time.time()
test_mels = np.stack(Parallel(n_jobs=-1)(delayed(test_mel)(t) for t in test["track_id"]))
labels = test["genre"].map({g: i for i, g in enumerate(CLASS_NAMES)}).to_numpy()
print(f"Spectrograms for {len(test_mels)} test songs in {time.time() - start:.0f}s")


@torch.no_grad()
def accuracy_after(seconds, batch=64):
    frames = int(seconds * SR / HOP) + 1
    preds = []
    for i in range(0, len(test_mels), batch):
        x = normalise(test_mels[i:i + batch, :, :frames].astype(np.float32))
        preds.append(model(x).argmax(1).cpu().numpy())
    return float((np.concatenate(preds) == labels).mean())


curve = pd.DataFrame({"seconds_heard": SECONDS, "test_accuracy": [accuracy_after(s) for s in SECONDS]})
curve.to_csv(RESULTS_DIR / "accuracy_vs_seconds.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["seconds_heard"], curve["test_accuracy"], marker="o")
ax.axhline(1 / len(CLASS_NAMES), ls="--", color="grey", label="random guess (12.5%)")
for s, a in zip(curve["seconds_heard"], curve["test_accuracy"]):
    ax.annotate(f"{a:.0%}", (s, a), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax.set(xlabel="seconds of audio heard", ylabel="test accuracy", ylim=(0, 1),
       title="C-RNN test accuracy after hearing only the first k seconds")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "accuracy_vs_seconds.png", dpi=150)
plt.show()
curve.style.format({"test_accuracy": "{:.2%}"})

## 8. Is it fast enough for real time?

Real time needs the model to process audio faster than it plays. The **real-time factor** is the
processing time divided by the audio length: below 1 means faster than real time.

In [ ]:
def timed(fn, repeats=20):
    fn()  # warm-up
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats


with torch.no_grad():
    mel_30 = test_mels[:1].astype(np.float32)
    model_time = timed(lambda: model(normalise(mel_30)))
y_30, _ = librosa.load(audio_path(test["track_id"].iloc[0]), sr=SR, mono=True, duration=CLIP_SECONDS)
mel_time = timed(lambda: mel_from_audio(y_30), repeats=5)

speed = pd.DataFrame({
    "step": ["mel-spectrogram (30 s clip)", "C-RNN forward pass (30 s clip)", "total"],
    "seconds": [mel_time, model_time, mel_time + model_time],
})
speed["real_time_factor"] = speed["seconds"] / CLIP_SECONDS
speed.to_csv(RESULTS_DIR / "speed.csv", index=False)
print("Device:", device)
speed.style.format({"seconds": "{:.4f}", "real_time_factor": "{:.4f}"})

## Notes

- **What this shows:** the model produces a usable genre estimate within the first seconds of a
  song, and processes audio far faster than it plays, which is what real-time classification needs.
- **Live use:** a streaming version would feed audio in short chunks and keep the GRU's hidden state
  between chunks; here the stream is simulated by cutting the audio at second k.
- **Limits:** look-ahead from the convolutions is under 0.2 s; accuracy on songs outside FMA's 8
  genres is not meaningful, because the model can only answer one of those 8.
- **Saved outputs:** `results/demo/` (live-prediction plot, accuracy vs. seconds, speed table).